### 3 Steps
- First step is to camera is stable or not 
- second step is to detect ROI by connected component analysis
- Block based approach is used 1 and 2 step
- Using K-temporal info decide it's smoke or not
br

In [2]:
import cv2
import numpy as np

# Open video file
cap = cv2.VideoCapture('../demoVideos/test (1).mp4')

while True:
    ret, frame = cap.read()
    frame = cv2.resize(frame, (960,540))
    
    if not ret:
        break
        
    cv2.imshow('Frames', frame)
    
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break   

# Release video capture object and clse all windows
cap.release()
cv2.destroyAllWindows()

In [26]:
# WX and WY (Sub-block size):
# These parameters define the size of the sub-blocks used for analysis.
# Larger values (e.g., 32x32) will make the algorithm 
# less sensitive to small movements efficient for computation

def camera_motion_detection(BG, BI, WX=20, WY=20, Th=0.5):
    # Calculate BD(t, x, y) as per equation (1)
    # Absolute block mean difference
    BD = np.abs(BG - BI)
    
    # Calculate sub-block average AD(i, j) as per equation (2)
    M, N = BD.shape
    i_blocks = M // WY
    j_blocks = N // WX
    
    AD = np.zeros((i_blocks, j_blocks))
    
    for i in range(i_blocks):
        for j in range(j_blocks):
            y_start, y_end = i*WY, (i+1)*WY
            x_start, x_end = j*WX, (j+1)*WX
            AD[i, j] = np.mean(BD[y_start:y_end, x_start:x_end])
    
    # Count sub-blocks with AD(i, j) > Th
    blocks_above_threshold = np.sum(AD > Th)
    total_blocks = i_blocks * j_blocks
    
    # Check if more than half of sub-blocks are above threshold
    if blocks_above_threshold > 0.9*total_blocks:
        return BI.copy(), BD, True
    else:
        # No significant camera motion, segment blobs
        return BG, BD, False
    

In [27]:
def blob_segmentation(BD, Prev_BB, Th=1):
    # Create binary image BB(t,x,y) as per equation (3)
    BB = (BD >= Th).astype(int)
    
    # Apply bitwise AND with previous frame's BB if available (equation 4)
    if prev_BB is not None:
        BB = np.logical_and(BB, prev_BB).astype(int)

#     # Apply morphological operations to remove small blobs
#     dilation_kernel = np.ones((2,2), dtype=bool)
#     erosion_kernel = np.ones((1,1), dtype=bool)
    
    BB = binary_dilation(BB)
    BB = binary_erosion(BB)
    
    # Apply connected component algorithm
    labeled_array, num_features = label(BB)
    
    return BB, labeled_array, num_features

In [29]:
import cv2
import numpy as np
from scipy.ndimage import label, binary_erosion, binary_dilation

# Open video file
BLOCK_SIZE = 4
prev_BB = None
cap = cv2.VideoCapture('../demoVideos/smoke.mp4')

# Initial frame processing
ret, frame = cap.read()
frame = cv2.resize(frame, (960,540))
yuv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
y_channel = yuv_frame[:, :, 0].astype(np.float32)
BG = cv2.boxFilter(y_channel, ddepth=-1, ksize=(BLOCK_SIZE, BLOCK_SIZE))

while True:
    ret, frame = cap.read()
    
    if not ret:
        break
        
    # Sequential Frame processing
    frame = cv2.resize(frame, (960,540))
    yuv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y_channel = yuv_frame[:, :, 0].astype(np.float32)
    BI = cv2.boxFilter(y_channel, ddepth=-1, ksize=(BLOCK_SIZE, BLOCK_SIZE))

    new_BG, BD, is_moving = camera_motion_detection(BG, BI)
    BG = new_BG
    
    cv2.imshow('Difference', BD)

    if not is_moving:
        BB, BLOBS, N = blob_segmentation(BD, Prev_BB = prev_BB)
        blobs_visual = (BLOBS.astype(np.float32) / BLOBS.max()) * 255
        cv2.imshow('BB Visualization', blobs_visual)
#         print('NO of BLOBs: ',N)
        prev_BB = BB
    else:
        prev_BB = None
    
    cv2.putText(frame, "motion", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0) if is_moving else (0, 0, 255), 2)
    
    cv2.imshow('Frames', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break   

# Release video capture object and clse all windows
cap.release()
cv2.destroyAllWindows()

In [4]:
cv2.destroyAllWindows()

### Smoke-detection approach using spatial and temporal analyses, which is based on the block-processing technique. This method analyzes energy-based and color-based features within the spatial, temporal, and spatial-temporal domains, before all the proposed features are combined using an SVM classifier

In [5]:
import cv2
import numpy as np

def detect_smoke_farneback(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, first_frame = cap.read()
    first_frame = cv2.resize(first_frame, (960,540))
    prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
    
    while True:
        ret, frame = cap.read()
        frame = cv2.resize(frame, (960,540))
        if not ret:
            break
        
        # Apply Gaussian blur to reduce noise
        frame_blur = cv2.GaussianBlur(frame, (5, 5), 0)
        gray = cv2.cvtColor(frame_blur, cv2.COLOR_BGR2GRAY)
        
        # Calculate dense optical flow using Farneback method
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 
                                            pyr_scale=0.5, levels=3, winsize=15, 
                                            iterations=3, poly_n=5, poly_sigma=1.1, flags=0)
        
        # Compute magnitude and angle of 2D vector
        magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        
        # Normalize magnitude for visualization
        magnitude_norm = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
        
        # Create a fresh mask for each frame
        mask = np.zeros_like(frame)
        mask[..., 1] = 255  # Set green channel to 255
        
        # Threshold for high magnitude
        high_magnitude_mask = magnitude > 0.1
        mask[..., 0][high_magnitude_mask] = angle[high_magnitude_mask] * 180 / np.pi / 2
        mask[..., 2][high_magnitude_mask] = magnitude_norm[high_magnitude_mask]
        
        # Convert HSV to RGB (BGR) color representation
        rgb = cv2.cvtColor(mask, cv2.COLOR_HSV2BGR)
        result = cv2.addWeighted(frame, 1, rgb, 2, 0)
     
        cv2.imshow('Smoke Detection Farneback', result)
        
        k = cv2.waitKey(30) & 0xff
        if k == 27:  # Press 'Esc' to exit
            break
        
        # Update previous frame
        prev_gray = gray
    
    cv2.destroyAllWindows()
    cap.release()

# Usage
video_path = '../demoVideos/smoke.mp4'
detect_smoke_farneback(video_path)


In [259]:
cv2.destroyAllWindows()
cap.release()

### Segmentation of Smoke Area!

In [6]:
import cv2
import numpy as np

# Open video file
cap = cv2.VideoCapture('../demoVideos/smoke.mp4')
fgbg = cv2.createBackgroundSubtractorMOG2()

while True:
    ret, frame = cap.read()
    frame = cv2.resize(frame, (960,540))
    
    if not ret:
        break
        
    fgmask = fgbg.apply(frame)
    cv2.imshow('Mask', fgmask)
        
    cv2.imshow('Frames', frame)
    
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break   

# Release video capture object and clse all windows
cap.release()
cv2.destroyAllWindows()

In [8]:
cv2.destroyAllWindows()

In [33]:
import cv2
import numpy as np
from scipy.ndimage import label, binary_erosion, binary_dilation

# Open video file
cap = cv2.VideoCapture('../demoVideos/smoke.mp4')

# Trackbar callback functions to update the values
def update_saturation(val):
    global saturation_thresh
    saturation_thresh = val

def update_th(val):
    global Th
    Th = val

# Create a window
cv2.namedWindow('Filtered Areas')

# Initial values for the thresholds
saturation_thresh = 64 # equivalent to 0.2 * 255
Th = 40

# Create trackbars for saturation threshold and Th
cv2.createTrackbar('Saturation Threshold', 'Filtered Areas', saturation_thresh, 255, update_saturation)
cv2.createTrackbar('Th', 'Filtered Areas', Th, 100, update_th)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.resize(frame, (960, 540))
    
    # Convert frame to HSV color space
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Split channels
    h, s, v = cv2.split(hsv)
    r, g, b = cv2.split(frame)
    
    # Create masks based on the given conditions
    mask1 = np.abs(r - g) < Th
    mask2 = np.abs(g - b) < Th
    mask3 = np.abs(r - b) < Th
    mask4 = s < saturation_thresh
    
    # Combine all masks
    combined_mask = mask1 & mask2 & mask3 & mask4
    combined_mask = binary_dilation(combined_mask, iterations=2)
    combined_mask = binary_erosion(combined_mask, iterations=2)
    
    # Create an output image that shows only the areas that meet the conditions
    output = np.zeros_like(frame)
    output[combined_mask] = frame[combined_mask]
    
    
    # Display the result
    cv2.imshow('Filtered Areas', output)
    cv2.imshow('Frames', frame)
    
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

# Release video capture object and close all windows
cap.release()
cv2.destroyAllWindows()
